In [ ]:
from analysis_helper import *

import warnings
from pathlib import Path
from matplotlib.patches import Patch

savefig = False

## Extract all scenarios

In [ ]:
# conference_v1 scenarios
scenarios = [
    "baseline-aims-3H", "decarbonize-aims-3H",
    "baseline-asean-3H", "decarbonize-asean-3H",
    "baseline-existing-3H", "decarbonize-existing-3H", 
]
years = [2025, 2030, 2035, 2040, 2045, 2050]

# sensitivity scenarios
# scenarios = [
#     "decarbonize-aims-3H",
#     "sensitivity-roof100-3H",
#     "sensitivity-roof200-3H",
#     "sensitivity-1H",
#     "sensitivity-3H",
#     "sensitivity-6H",
#     "sensitivity-50n-3H",
#     "sensitivity-200n-3H",
# ]
# years = [2050]

df_scene = pd.DataFrame(index=scenarios, columns=years)
for scenario in df_scene.index:
    for year in df_scene.columns:
        base_path = Path(f"../results/{scenario}/postnetworks/")
        try:
            fn = [p for p in base_path.rglob("*.nc") if str(year) in p.name][0]
            n = pypsa.Network(fn)
            n = clean_carrier(n)
            df_scene.loc[scenario,year] = n
        except:
            df_scene.loc[scenario,year] = np.nan
            break # Break one level loop

# Generate KPI Tables

In [ ]:
def get_total_energy_traded(n0):

    # Strip network
    n = strip_network(n0)
    
    lines = n.lines.copy()
    links = n.links.copy()
    
    lines["country0"] = [n.buses.country[busname] for busname in n.lines.bus0]
    lines["country1"] = [n.buses.country[busname] for busname in n.lines.bus1]
    lines_index = lines.query('country0 != country1').index
    
    lines_total = (n.snapshot_weightings.generators @ abs(n.lines_t.p0[lines_index])).sum()
    
    links["country0"] = [n.buses.country[busname] for busname in n.links.bus0]
    links["country1"] = [n.buses.country[busname] for busname in n.links.bus1]
    links_index = links.query('country0 != country1 & carrier != "DC"').index
    
    links_total = (n.snapshot_weightings.generators @ abs(n.links_t.p0[links_index])).sum()
    
    return lines_total + links_total

res_carriers = [
    'biomass',         # Dispatchable, renewable, relatively stable
    'biomass EOP',
    'geothermal',      # Very stable, constant output
    'hydro',           # Controllable to some extent, seasonal
    'ror',             # Run-of-river, weather-dependent
    'offwind-dc',      # Offshore wind, variable but often more stable
    'offwind-ac',      # Same as above, different transmission
    'onwind',          # Land-based wind, variable
    'solar',           # Daylight only, weather-sensitive
    'solar rooftop'    # Most variable, decentralized, small-scale
]

load = df_scene.copy(deep=True)
system_cost = df_scene.copy(deep=True)
emission = df_scene.copy(deep=True)
res_gen = df_scene.copy(deep=True)
marginal_price = df_scene.copy(deep=True)
trade = df_scene.copy(deep=True)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning)

    for scenario in scenarios:
            for year in years:
                n = df_scene.loc[scenario, year]
                if isinstance(n, pypsa.Network):
                    # Calculate Total Load
                    load.loc[scenario, year] = - n.statistics.energy_balance(comps=["Load"]).sum()
                    
                    # Calculate System Cost
                    system_cost.loc[scenario, year] = n.statistics.system_cost().sum()
        
                    # Calculate Emissions
                    emission.loc[scenario, year] = n.stores_t.e["co2 atmosphere"].iloc[-1]
        
                    # Calculate Renewable generation
                    df_res = pd.DataFrame(n.statistics.energy_balance(nice_names=False)).reset_index()
                    res_gen.loc[scenario, year]  =  df_res[
                        df_res.carrier.isin(res_carriers) & df_res.bus_carrier.isin(["AC","DC","low voltage"])
                    ][0].sum()
        
                    marginal_price.loc[scenario, year] = n.statistics.prices(groupby='bus_carrier')["low voltage"]
                    
                    trade.loc[scenario, year] = get_total_energy_traded(n)
                else:
                    continue
                    #break

emission_intensity = emission / load
res_share = res_gen / load

# Ensure all data is numeric
system_cost = (system_cost.apply(pd.to_numeric, errors='coerce') / 1e9).round(2)
marginal_price = (marginal_price.apply(pd.to_numeric, errors='coerce')).round(2)
emission = (emission.apply(pd.to_numeric, errors='coerce') / 1e6).round(2)
emission_intensity = (emission_intensity.apply(pd.to_numeric, errors='coerce') * 1e3).round(2)
trade = (trade.apply(pd.to_numeric, errors='coerce') / 1e6).round(2)
res_gen = (res_gen.apply(pd.to_numeric, errors='coerce') / 1e6).round(2)
res_share = (res_share.apply(pd.to_numeric, errors='coerce') * 100).round(2)

dfs = {
    "System Cost [Billion EUR/a]": system_cost,
    "Marginal Cost [EUR/MWh]": marginal_price,
    # "Emissions [Mton CO2eq/a]": emission,
    "Emission Intensity [gCO2/kWh]": emission_intensity,
    "Energy Traded [TWh]": trade,
    "Renewable Generation [TWh]": res_gen,
    "Renewable Share [%]": res_share,
}

results = pd.concat(
    dfs,
    axis=0,
    names=["Metric", system_cost.index.name or "Index"]
)

# results.to_csv("results_summary.csv")
results

# pd.DataFrame(results.values).set_index(0)

In [ ]:
simplify_network = {
    "baseline-existing-3H":"Baseline",
    "decarbonize-existing-3H":"Dec. Existing",
    "decarbonize-aims-3H":"Dec. AIMS",
}

results_s = results[results.index.get_level_values("Index").isin(simplify_network.keys())][2050].unstack("Index")
results_s.columns = results_s.columns.map(simplify_network)
results_s = results_s[["Baseline","Dec. Existing","Dec. AIMS"]]
results_s

# pd.DataFrame(results_s.values).set_index(0)

# Generate Energy Network Map

In [ ]:
def plot_generation_ax(df, name, ax, years=[2025, 2030, 2035, 2040, 2045, 2050]):
    
    # ---- Data aggregation ----
    df_energy_all = pd.DataFrame(columns=years)
    df_load_all = pd.DataFrame(columns=years)
    df_store_all = pd.DataFrame(columns=years)

    for year in years:
        n = df.loc[name, year]

        df_energy = pd.DataFrame(
            n.statistics.energy_balance(
                comps=['Generator', 'Link', 'StorageUnit', 'Store'],
                nice_names=False
            )
        ).reset_index()

        df_energy = df_energy[
            ~df_energy.carrier.isin(elec_bus_carrier) &
            df_energy.bus_carrier.isin(elec_bus_carrier)
        ]

        df_energy_all[year] = df_energy.groupby("carrier").sum()[0]

        df_load_all.loc["Demand", year] = -n.statistics.energy_balance(
            comps=["Load"]
        ).sum() / 1e6

        store_bus = n.stores[n.stores.carrier.isin(
            ['H2 Store Tank', 'battery', 'home battery']
        )].index

        df_store_all.loc["Storage Discharge", year] = (
            - n.snapshot_weightings.objective @ n.stores_t.p[store_bus].clip(upper=0)
        ).sum() / 1e6

    df_energy_all = df_energy_all.rename(index=battery_pair).groupby("carrier").sum() / 1e6

    df_pos = df_energy_all.T.clip(lower=0)
    df_pos = df_pos.loc[:, df_pos.sum(skipna=True) > 1]

    ordered_carrier = [c for c in order_carrier if c in df_pos.columns]
    remaining_cols = [c for c in df_pos.columns if c not in order_carrier]
    df_pos = df_pos.reindex(columns=ordered_carrier + remaining_cols)

    # ---- Plotting ----

    # Storage discharge
    df_store_all.T.plot(
        ax=ax,
        color="blue",
        linewidth=2,
        style='--',
        alpha=0.8,
        legend=False
    )

    # Generation area
    df_pos.plot(
        kind='area',
        stacked=True,
        ax=ax,
        color=[n.carriers.color[c] for c in df_pos.columns],
        linewidth=0,
        alpha=0.8,
        legend=False
    )

    # Load demand
    df_load_all.T.plot(
        ax=ax,
        color="black",
        linewidth=2,
        style='--',
        alpha=0.8,
        legend=False
    )

    ax.set_title(name)
    ax.set_ylabel("Energy [TWh]")
    ax.set_xlabel("Year")
    ax.set_xlim([years[0], years[-1]])
    ax.set_ylim([0, 3500])
    ax.grid(axis='y')

    return ax

def plot_generation(df, name, years=[2025, 2030, 2035, 2040, 2045, 2050]):

    fig, ax = plt.subplots(figsize=(8, 8))

    ax = plot_generation_ax(df, name, ax, years=years)

    df.iloc[0,0]

    # Reverse the legend order
    handles, labels = ax.get_legend_handles_labels()
    handles, labels = handles[::-1], labels[::-1]

    labels = [n.carriers.nice_name[lable] if lable in n.carriers.nice_name else lable for lable in labels]
    ax.legend(handles, labels, title="Carrier", bbox_to_anchor=(1.05, 1.1), loc='upper left')

    plt.tight_layout()
    plt.show()


def plot_costs_ax(df, name, ax, years=[2025, 2030, 2035, 2040, 2045, 2050], title=True):

    # --- Rename carriers ---
    mapping = {
        ('Generator', 'lignite'): ('Generator', 'lignite fuel'),
        ('Generator', 'coal'): ('Generator', 'coal fuel'),
        ('Generator', 'gas'): ('Generator', 'gas fuel'),
    }

    hatched_carriers = {"lignite fuel", "coal fuel", "gas fuel", "solid biomass"}

    # ---- Data aggregation ----
    df_system_cost_all = pd.DataFrame(columns=years)

    for year in years:
        n = df.loc[name, year]

        costs = n.statistics.system_cost(nice_names=False)
        costs.index = costs.index.map(lambda x: mapping.get(x, x))

        df_system_cost_all[year] = costs.groupby("carrier").sum()

    df_system_cost_all = (
        df_system_cost_all
        .rename(index=battery_pair)
        .groupby("carrier")
        .sum() / 1e9
    )

    # ---- Prepare plotting data ----
    df_pos = df_system_cost_all.T.clip(lower=0)
    df_pos = df_pos.loc[:, df_pos.sum(skipna=True) > 1]

    ordered_carrier = [c for c in order_carrier if c in df_pos.columns]
    remaining_cols = [c for c in df_pos.columns if c not in order_carrier]
    df_pos = df_pos.reindex(columns=ordered_carrier + remaining_cols)

    # ---- Plot ----
    df_pos.plot(
        kind="area",
        stacked=True,
        ax=ax,
        color=[n.carriers.color[c] for c in df_pos.columns],
        linewidth=0,
        alpha=0.8,
        legend=False,
    )

    # ---- Apply hatching ----
    for poly, carrier in zip(ax.collections, df_pos.columns):
        if carrier in hatched_carriers:
            poly.set_hatch("\\")
            poly.set_edgecolor("k")
            poly.set_linewidth(0.3)

    # ---- Labels and aesthetics ----
    if title:
        ax.set_title(name)
    ax.set_ylabel("System Cost [Billion €/a]")
    ax.set_xlabel("Year")
    ax.set_xlim([years[0], years[-1]])
    ax.set_ylim([0, 300])
    ax.grid(axis="y")

    return ax


def plot_system_cost(df, name, years=[2025, 2030, 2035, 2040, 2045, 2050]):

    fig, ax = plt.subplots(figsize=(8, 8))
    
    ax = plot_costs_ax(df, name, ax, years=years)
    
    # Reverse the legend order
    handles, labels = ax.get_legend_handles_labels()
    handles, labels = handles[::-1], labels[::-1]
    
    labels = [n.carriers.nice_name[lable] if lable in n.carriers.nice_name else lable for lable in labels]
    ax.legend(handles, labels, title="Carrier", bbox_to_anchor=(1.05, 1.1), loc='upper left')
    
    plt.tight_layout()
    plt.show()


def plot_map_ax(
    n, 
    ax,
    bus_factor = 2e8,
    width_factor = 5e7,
    flow_factor = 7e2
):
    
    # Strip network
    m = strip_network(n)
    
    # Compute flows
    line_flow = m.snapshot_weightings.objective @ m.lines_t.p0
    link_flow = m.snapshot_weightings.objective @ m.links_t.p0
    


    bus_carrier = get_energy_balance_bus(n) / bus_factor
    
    m.plot(
        ax=ax,
        bus_sizes= bus_carrier,
        line_colors="black",
        link_colors="black",
        line_widths=line_flow / width_factor,
        link_widths=link_flow / width_factor,
        line_flow=np.sign(line_flow) * flow_factor,
        link_flow=np.sign(link_flow) * flow_factor,
    )

    # -----------------------
    # Carrier color legend
    # -----------------------
    focus_carrier = [k for k, v in bus_carrier.groupby("carrier").sum().items() if v > 0]
    ordered_carrier = [c for c in order_carrier if c in focus_carrier]
    carrier_list = m.carriers.loc[ordered_carrier,["color","nice_name"]].set_index("nice_name")

    return ax, carrier_list

def add_default_legend(
    fig, 
    ax,
    carrier_list,
    bus_factor = 2e8,
    width_factor = 5e7,
    flow_factor = 7e2
):

    carrier_handles = [
        Patch(facecolor=color, label=carrier)
        for carrier, color in carrier_list.color.items()
    ]
    
    fig.legend(
        handles=carrier_handles,
        title="Carrier",
        loc="center right",
        alignment="center",
        bbox_to_anchor=(1.08, 0.5),
        frameon=True
    )

    legend_bus = {
        "sizes": [100 * 1e6 / bus_factor, 50 * 1e6 / bus_factor, 25 * 1e6 / bus_factor],
        "labels": ["100 TWh", "50 TWh", "25 TWh"]
    }
    
    pypsa.plot.add_legend_circles(
        ax=ax, 
        sizes = legend_bus["sizes"], 
        labels = legend_bus["labels"],
        patch_kw = {'color':'silver'},
        legend_kw = {'loc':"center right",
                     'bbox_to_anchor':(1.01, 0.85),
                     'frameon':False, 
                     'labelspacing':1,
                     'title':"Generation",
                    }
    )

    legend_line = {
        "sizes": [100 * 1e6 / width_factor, 50 * 1e6 / width_factor, 25 * 1e6 / width_factor],
        "labels": ["100 TWh", "50 TWh", "25 TWh"]
    }
    
    pypsa.plot.add_legend_lines(
        ax=ax, 
        sizes = legend_line["sizes"], 
        labels = legend_line["labels"],
        patch_kw = {'color':"black"},
        legend_kw = {'loc':"center right",
                     'bbox_to_anchor':(0.99, 0.5),
                     'frameon':False,
                     'labelspacing':1,
                     'title':"Energy Flow",
                    }
    )

    
    plt.subplots_adjust(right=0.8)

    return fig, ax


In [ ]:
scenario = "decarbonize-aims-3H"
year = 2050

fig, ax = plt.subplots(
    figsize=(8, 8),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

ax, carrier_list = plot_map_ax(
    df_scene.loc[scenario, year],
    ax
)

fig, ax = add_default_legend(fig, ax, carrier_list.iloc[::-1])

ax.set_title(f"{scenario} - Year {year}")

if savefig:
    fig.savefig(
        f"{scenario}-{year}-map.png",
        dpi=300,
        bbox_inches="tight",
    )

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

year = 2050

fig, ax = plt.subplots(
    2,1,
    figsize=(8, 12),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

ax[0], _ = plot_map_ax(
    df_scene.loc["baseline-existing-3H", year],
    ax[0]
)

ax[1], carrier_list = plot_map_ax(
    df_scene.loc["decarbonize-existing-3H", year],
    ax[1]
)

ax[0].set_title(f"baseline-existing-3H - Year {year}")
ax[1].set_title(f"decarbonize-existing-3H - Year {year}")

carrier_handles = [
    Patch(facecolor=color, label=carrier)
    for carrier, color in carrier_list.iloc[::-1].color.items()
]

fig.legend(
    handles=carrier_handles,
    title="Carrier",
    loc="center right",
    alignment="center",
    bbox_to_anchor=(1.17, 0.5),
    frameon=True
)

bus_factor = 2e8
width_factor = 5e7
flow_factor = 7e2

legend_bus = {
    "sizes": [100 * 1e6 / bus_factor, 50 * 1e6 / bus_factor, 25 * 1e6 / bus_factor],
    "labels": ["100 TWh", "50 TWh", "25 TWh"]
}

pypsa.plot.add_legend_circles(
    ax=ax[0], 
    sizes = legend_bus["sizes"], 
    labels = legend_bus["labels"],
    patch_kw = {'color':'silver'},
    legend_kw = {'loc':"center right",
                 'bbox_to_anchor':(1.3, 0.53),
                 'frameon':True, 
                 'labelspacing':1,
                 'title':"Generation",
                }
)

legend_line = {
    "sizes": [100 * 1e6 / width_factor, 50 * 1e6 / width_factor, 25 * 1e6 / width_factor],
    "labels": ["100 TWh", "50 TWh", "25 TWh"]
}

pypsa.plot.add_legend_lines(
    ax=ax[1], 
    sizes = legend_line["sizes"], 
    labels = legend_line["labels"],
    patch_kw = {'color':"black"},
    legend_kw = {'loc':"center right",
                 'bbox_to_anchor':(1.3, 0.5),
                 'frameon':True,
                 'labelspacing':1,
                 'title':"Energy Flow",
                }
)

fig.subplots_adjust(hspace=0.05)

if savefig:
    fig.savefig(
        f"summary-{year}-map.png",
        dpi=300,
        bbox_inches="tight",
    )

In [ ]:
scenario = "baseline-existing-3H"
year = 2050

fig, ax = plt.subplots(
    figsize=(8, 8),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

ax, carrier_list = plot_map_ax(
    df_scene.loc[scenario, year],
    ax
)

fig, ax = add_default_legend(fig, ax, carrier_list.iloc[::-1])

ax.set_title(f"{scenario} - Year {year}")

if savefig:
    fig.savefig(
        f"{scenario}-{year}-map.png",
        dpi=300,
        bbox_inches="tight",
    )

# Generate Total Energy Mix and System Cost Figures

In [ ]:
def combined_figure_legend(fig, loc="upper center", ncol=1, **legend_kwargs):
    """
    Create a combined legend for all axes in a figure without duplicate labels.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Figure containing multiple axes.
    loc : str, optional
        Legend location.
    ncol : int, optional
        Number of legend columns.
    legend_kwargs : dict
        Additional keyword arguments passed to fig.legend().
    """
    from collections import OrderedDict
    
    handles, labels = [], []

    # Collect legend entries from all axes
    for ax in fig.axes:
        h, l = ax.get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)

    # De-duplicate while preserving first occurrence
    unique = OrderedDict(zip(labels, handles))

    # Split into ordered and unordered labels
    ordered_handles = []
    ordered_labels = []

    for label in order_carrier:
        if label in unique:
            ordered_labels.append(label)
            ordered_handles.append(unique[label])

    # Append labels not in order_carrier
    for label, handle in unique.items():
        if label not in order_carrier:
            ordered_labels.append(label)
            ordered_handles.append(handle)

    ordered_handles = ordered_handles[::-1]
    ordered_labels = ordered_labels[::-1]

    nice_name = n.carriers.nice_name.map(nice_name_rename).fillna(n.carriers.nice_name)
    ordered_labels = [nice_name[lable] if lable in nice_name else lable for lable in ordered_labels]

    fig.legend(
        ordered_handles,
        ordered_labels,
        loc=loc,
        ncol=ncol,
        bbox_to_anchor=(0.5, -0.15),
        **legend_kwargs
    )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True)

axes[0, 1].sharey(axes[0, 0])  # top row
axes[1, 1].sharey(axes[1, 0])  # bottom row

#for ax, scenario in zip(axes.flat, df_scene.index):
plot_generation_ax(df_scene, 'baseline-aims-3H', ax=axes[0,0])
plot_generation_ax(df_scene, 'decarbonize-aims-3H', ax=axes[0,1])
plot_costs_ax(df_scene, 'baseline-aims-3H', ax=axes[1,0], title=False)
plot_costs_ax(df_scene, 'decarbonize-aims-3H', ax=axes[1,1], title=False)

combined_figure_legend(fig, loc="lower center", ncol=4, title="Carrier")

# --- Make room for legend ---
plt.subplots_adjust(bottom=0.13)   # increase space at bottom
plt.subplots_adjust(wspace=0.1, hspace=0.1)

if savefig:
    fig.savefig(
        "aims-energy-cost-3H.png",
        dpi=300,
        bbox_inches="tight",
    )

plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True)

axes[0, 1].sharey(axes[0, 0])  # top row
axes[1, 1].sharey(axes[1, 0])  # bottom row

#for ax, scenario in zip(axes.flat, df_scene.index):
plot_generation_ax(df_scene, 'baseline-existing-3H', ax=axes[0,0])
plot_generation_ax(df_scene, 'decarbonize-existing-3H', ax=axes[0,1])
plot_costs_ax(df_scene, 'baseline-existing-3H', ax=axes[1,0], title=False)
plot_costs_ax(df_scene, 'decarbonize-existing-3H', ax=axes[1,1], title=False)

combined_figure_legend(fig, loc="lower center", ncol=4, title="Carrier")

# --- Make room for legend ---
plt.subplots_adjust(bottom=0.13)   # increase space at bottom
plt.subplots_adjust(wspace=0.1, hspace=0.1)

if savefig:
    fig.savefig(
        "existing-energy-cost-3H.png",
        dpi=300,
        bbox_inches="tight",
    )

plt.show()